In [1]:
import pyspark 
import pandas as pd
from pyspark.sql import SparkSession,Row,DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import *
from config.settings import Config


spark=SparkSession.builder.master("local[1]")\
        .appName("Netflix")\
        .config("spark.sql.warehouse.dir", "/home/jovyan/work/database") \
        .config("spark.jars", "/Users/eduardoalberto/opt/spark-4.0.0/jars/postgresql-42.7.3.jar")\
        .enableHiveSupport()\
        .getOrCreate()
sc = spark.sparkContext
spark.sparkContext.setLogLevel("OFF") 
print('PySpark Version :'+spark.version)
print('PySpark Version :'+spark.sparkContext.version)

spark

25/09/24 22:38:08 WARN Utils: Your hostname, MacBook-Pro-de-Eduardo.local resolves to a loopback address: 127.0.0.1; using 192.168.3.178 instead (on interface en0)
25/09/24 22:38:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/eduardoalberto/.ivy2/cache
The jars for the packages stored in: /Users/eduardoalberto/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-805d5991-95d1-4dc4-bb10-009f6e316815;1.0
	confs: [default]


:: loading settings :: url = jar:file:/Users/eduardoalberto/opt/spark-3.5.4/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 in central
	found org.mongodb#mongodb-driver-sync;4.8.2 in central
	[4.8.2] org.mongodb#mongodb-driver-sync;[4.8.1,4.8.99)
	found org.mongodb#bson;4.8.2 in central
	found org.mongodb#mongodb-driver-core;4.8.2 in central
	found org.mongodb#bson-record-codec;4.8.2 in central
:: resolution report :: resolve 2971ms :: artifacts dl 3ms
	:: modules in use:
	org.mongodb#bson;4.8.2 from central in [default]
	org.mongodb#bson-record-codec;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-core;4.8.2 from central in [default]
	org.mongodb#mongodb-driver-sync;4.8.2 from central in [default]
	org.mongodb.spark#mongo-spark-connector_2.12;10.3.0 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	----------------------------------------------------

PySpark Version :3.5.4
PySpark Version :3.5.4


In [2]:
df = Config.CSV_OPTIONS["arq"]

spark.read.csv(df, header=True, inferSchema=True).show(5)


+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|show_id|   type|               title|       director|                cast|      country|        date_added|release_year|rating| duration|           listed_in|         description|
+-------+-------+--------------------+---------------+--------------------+-------------+------------------+------------+------+---------+--------------------+--------------------+
|     s1|  Movie|Dick Johnson Is Dead|Kirsten Johnson|                NULL|United States|September 25, 2021|        2020| PG-13|   90 min|       Documentaries|As her father nea...|
|     s2|TV Show|       Blood & Water|           NULL|Ama Qamata, Khosi...| South Africa|September 24, 2021|        2021| TV-MA|2 Seasons|International TV ...|After crossing pa...|
|     s3|TV Show|           Ganglands|Julien Leclercq|Sami Bouajila, Tr...|         NULL|Septem

In [ ]:
spark.read.csv('/Users/eduardoalberto/LoadFile/input/netflix_titles_clean.csv',header=True,inferSchema=True,sep="," ,quote= '"',escape='"')\
            .createOrReplaceTempView("tb_netflix_titles")

In [ ]:
# !pip install --upgrade "pandas>=2.0.0"

#!pip install --upgrade pyspark==4.0.0
#!pip3.9 install pyspark==3.5.3
!pip3.9 list


In [ ]:
# spark.table("tb_netflix_titles").filter(F.col("date_added") == "Toni Tones").show()


df = (
    spark.table("tb_netflix_titles")
        .withColumn("id", F.substring(F.col("show_id"), 2, 4))
        .withColumn("date_added", F.trim(F.col("date_added")))
        # só converte valores que correspondem ao padrão de data
        .withColumn(
            "dt_added",
            F.when(
                F.col("date_added").rlike("^[A-Za-z]+ [0-9]{1,2}, [0-9]{4}$"),
                F.to_date(F.col("date_added"), "MMMM d, yyyy")
            ).otherwise(None)
        )
)




df.write.mode("overwrite").parquet("/Users/eduardoalberto/LoadFile/repository/teste/")
df.toPandas()




In [ ]:
spark.table("tb_netflix_titles").select("rating").distinct().toPandas()

In [ ]:
! ls /Users/eduardoalberto/LoadFile/output/netflix/processados/arq/data_execucao=2025-08-25



In [ ]:
df = spark.read.parquet("/Users/eduardoalberto/LoadFile/output/netflix/processados/arq/data_execucao=2025-08-25/*.parquet")

df.toPandas()



### valida DQ

In [ ]:
import os
props = {
    "user": os.getenv("DB_USER", "dbpostgres"),
    "password": os.getenv("DB_PASSWORD", "postgre123"),
    "driver": "org.postgresql.Driver"
}

spark.read.jdbc("jdbc:postgresql://localhost:5432/dbpostgres", "data_quality_report",properties=props).createOrReplaceTempView("tb_dq_report")
spark.table("tb_dq_report").select("coluna").distinct().toPandas()
